## AP pillar backfill (one-off, standalone)

`run_260727_0922`'s LLM labelling already completed before REVIEW POINT 5 (AP pillar
normalization) existed in S2, so `funding_curated`'s pre-existing rows still have their old,
messy `AP pillar` values - including 22 `'Agnostic'` and 9 blank rows that were never confidently
resolvable automatically. Because `build_pool_b` in `S6_LLM_labelling.py` only labels rows whose
`AP pillar` maps to one of the 4 canonical pillar codes, these rows were silently skipped for
`rescat` labelling (and always will be, until their pillar is fixed).

This notebook fixes that directly on the **live** `funding_curated` table and then runs `rescat`
labelling for just those rows - without re-running S2's full rebuild cell, which would wipe out
the 611 Dimensions grants `run_260727_0922` already promoted into `funding_curated`.

Steps:
1. Auto-normalize `AP pillar` in place (casing, meat/seafood suffix, multi-pillar combos) -
   same rules as S2's REVIEW POINT 5, reused from `Funding_dedup_helpers`.
2. Export whatever's still unresolved (`Agnostic` + blank) for manual review.
3. *(pause - fill in the exported CSV)*
4. Apply your reviewed values back onto the live table.
5. Run `rescat`-only, `new_only`-scope labelling via `S6_LLM_labelling.main()` - this submits a
   real (billed) Batch API request and writes the results straight into `funding_curated`.

Not part of the regular pipeline - safe to delete once run.

In [1]:
import duckdb
import pandas as pd
from pathlib import Path

from Funding_dedup_helpers import normalize_ap_pillar, export_for_review, apply_reviewed_values

DB_PATH = Path('funding.db')
DATA_DIR = Path('data_review')
DATA_DIR.mkdir(exist_ok=True)

# The run whose promoted grants prompted this one-off fix - only used to name the review CSV and
# the rescat batch, so this stays traceable to that run alongside its other outputs.
RUN_TIMESTAMP = '260727_0922'

### 1-2. Auto-normalize + export what's left for review

In [2]:
con = duckdb.connect(str(DB_PATH))
curated_df = con.sql("SELECT * FROM funding_curated").df()

original_ap_pillar = curated_df['AP pillar'].copy()
_normalized = original_ap_pillar.apply(normalize_ap_pillar)
new_ap_pillar = _normalized.apply(lambda t: t[0])
resolved_mask = _normalized.apply(lambda t: t[1])

# Only rows sourced from the last report / Grants Tracker (lrd_row_id not null) predate REVIEW
# POINT 5 - promoted Dimensions grants already got a canonical AP pillar at promotion time.
has_lrd_id = curated_df['lrd_row_id'].notna()
changed_mask = resolved_mask & (new_ap_pillar != original_ap_pillar) & has_lrd_id

changed_df = curated_df.loc[changed_mask, ['lrd_row_id']].copy()
changed_df['AP pillar'] = new_ap_pillar[changed_mask]

if len(changed_df):
    con.register('changed_df', changed_df)
    con.sql('''
        UPDATE funding_curated
        SET "AP pillar" = changed_df."AP pillar"
        FROM changed_df
        WHERE funding_curated.lrd_row_id = changed_df.lrd_row_id
    ''')
    con.unregister('changed_df')
print(f"Auto-normalized and wrote back {len(changed_df)} 'AP pillar' values.")

review_mask = (~resolved_mask) & has_lrd_id
ap_pillar_review_df = curated_df.loc[
    review_mask, ['lrd_row_id', 'Identification code', 'Title', 'Abstract', 'AP pillar']
].reset_index(drop=True)

ap_pillar_review_df = export_for_review(
    ap_pillar_review_df,
    id_cols=['lrd_row_id'],
    out_path=DATA_DIR / f'{RUN_TIMESTAMP}_ap_pillar_for_review.csv',
    decision_col='corrected_ap_pillar',
    default='',
)

Auto-normalized and wrote back 0 'AP pillar' values.
Saved 31 candidate matches for review -> data_review\260727_0922_ap_pillar_for_review.csv


### 3. Pause here

Open `data_review/{RUN_TIMESTAMP}_ap_pillar_for_review.csv`, fill in `corrected_ap_pillar` for
every row with one of `Plant-based` / `Fermentation` / `Cultivated` / `Cross-cutting`, save it,
then continue with the next cell.

In [3]:
reviewed_df = apply_reviewed_values(
    ap_pillar_review_df,
    DATA_DIR / f'{RUN_TIMESTAMP}_ap_pillar_for_review.csv',
    id_cols=['lrd_row_id'],
    value_col='corrected_ap_pillar',
)

con.register('reviewed_df', reviewed_df[['lrd_row_id', 'corrected_ap_pillar']])
con.sql('''
    UPDATE funding_curated
    SET "AP pillar" = reviewed_df.corrected_ap_pillar
    FROM reviewed_df
    WHERE funding_curated.lrd_row_id = reviewed_df.lrd_row_id
''')
con.unregister('reviewed_df')
print(f"Wrote {len(reviewed_df)} manually-reviewed 'AP pillar' values back to funding_curated.")

con.close()

Wrote 31 manually-reviewed 'AP pillar' values back to funding_curated.


### 4. Run rescat-only labelling for the newly-fixed rows

`build_pool_b` in S6 only requires a valid `AP pillar` and (for `new_only`) a blank `Research
category` - both now true for exactly the rows just fixed above, so this submits a batch for just
them (plus, harmlessly, any other genuinely-unlabelled row - none are expected right now since
`run_260727_0922` already relabelled everything else with `LABEL_SCOPE='all'`).

In [4]:
from S6_LLM_labelling import main as run_llm_labelling

run_llm_labelling(
    RUN_RESCAT=True,
    RUN_ENDPRODUCT=False,
    RUN_AWARDPURPOSE=False,
    RUN_SUBPILLAR=False,
    LABEL_SCOPE='new_only',
    RUN_LABEL=f'{RUN_TIMESTAMP}_ap_pillar_backfill',
)


Connecting to the Anthropic API
[rescat] Pool A (funding_classified): 0 rows
[rescat] Pool B (funding_curated): 71 rows
[rescat] Submitting batch of 71 requests
[rescat] Batch ID: msgbatch_01CDVxfgVffbVi5t2qN1nL4F   Status: in_progress

Waiting for all submitted batches to complete
[rescat] status: in_progress   counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=71, succeeded=0)
[rescat] status: ended   counts: MessageBatchRequestCounts(canceled=0, errored=0, expired=0, processing=0, succeeded=71)

Retrieving results
[rescat] Pool A results: 0 total, 0 succeeded
[rescat] Pool B results: 71 total, 71 succeeded
No new Dimensions grants to promote.
'funding_curated' rewritten with 2489 total rows.
Excel copy of 'funding_curated' saved to data_output\260727_0922_ap_pillar_backfill_funding_curated.xlsx
No failed labelling attempts this run.

Done!
